### pyAPES_MLM: Multi-layer soil-vegetation-atmosphere transfer model

Demo how to:
1. Set-up the model
1. Run the model
1. Retrieve the results using xarray

In [ ]:
# setting path
import sys
import os

from dotenv import load_dotenv
load_dotenv()
pyAPES_main_folder = os.getenv('pyAPES_main_folder')
sys.path.append(pyAPES_main_folder)
os.chdir(pyAPES_main_folder)

### Import modules

In [ ]:
# function to read forcing data. See 'forcing/forcing_info.txt' for model forcing variable names and units!
from pyAPES.utils.iotools import read_forcing

# import the multi-layer model (mlm) driver
from pyAPES.pyAPES_MLM import driver

eps = 1e-16

### Read model parameter dictionaries

- mlm setup (parameterization, forcing file & run info) defined in: pyAPES.models.mlm_parameters
- output variables & logger configuration defined in: pyAPES.models.mlm_outputs. This is read by driver

In [ ]:
# import parameter dictionaries
from pyAPES.parameters.mlm_parameters import gpara, cpara, spara # model configuration, canopy parameters, soil parameters

### Read forcing data and compile input for driver

- forcing is read into pd.DataFrame. Units must be correct!
- create parameter dictionary and provide it to driver

In [ ]:
gpara['start_time'] = '2006-10-01'
gpara['end_time'] = '2007-06-01'

cpara['Ebal'] = False
snowmod = 'degreeday' # 'fsm2' or 'degreeday'
cpara['forestfloor']['snowpack']['snow_model'] = snowmod # 'fsm2' or 'degreeday'

forcing = read_forcing(
    forcing_file=gpara['forc_filename'],
    start_time=gpara['start_time'],
    end_time=gpara['end_time'],
    dt=gpara['dt']
)

params = {
    'general': gpara,   # model configuration
    'canopy': cpara,    # planttype, micromet, canopy, bottomlayer parameters
    'soil': spara,      # soil heat and water flow parameters
    'forcing': forcing  # forging data
}

In [ ]:
resultfile, Model = driver(parameters=params,
                           create_ncf=True,
                           result_file= 'testrun.nc'
                          )

# alternative (for testing over short periods) is to return resuts in dictionary 'results'
# results, Model = driver(parameters=params,
#                         create_ncf=False,
#                         result_file=None
#                          )

### Run the model

In [ ]:
#resultfile, Model = driver(parameters=params,
#                           create_ncf=True,
#                           result_file= 'testrun.nc'
#                          )

# alternative (for testing over short periods) is to return resuts in dictionary 'results'
#results, Model = driver(parameters=params,
#                         create_ncf=False,
#                         result_file=None
#                          )

### Model run completed and results are in NetCDF-file.

### Read results using xarray
- we use function 'read_results' from pyAPES.utils.utils

- see https://docs.xarray.dev/en/stable/



In [ ]:
from pyAPES.utils.iotools import read_results
import matplotlib.pyplot as plt

# read simulation restuls to xarray dataset
results = read_results(resultfile)

### Print resuts metadata:

Output variables are those defined at pyAPES.parameters.mlm_outputs'. Each variable contains one or several of following dimensions:

- date --> time dimension
- simulation --> simulation number. If there is only one, remember that it has index 0!
- canopy --> canopy and air layers. Index is 0 at ground and increases upwards
- planttype --> planttypes defined in mlm_parameters. Index follows the order in dict cpara['planttypes']) 
- soil -> soil layers. Index 0 is at the top and increases downwards (nr soil layers, 0 at top)
- groundtype --> groundtypes at forestfloor. Defined in mlm_parameters. Index follows the order in dict: cpara['forestfloor']['bottom_layer_types']


In [ ]:
print(results)

# print list of all variables:
#vars = list(results.data_vars)
#for v in vars:
#    print(v)

In [ ]:
results['ffloor_snow_depth'][:,0].plot()

In [ ]:
results['ffloor_snow_water_equivalent'].plot()
(results['ffloor_snow_depth']*100).plot()
#results['ffloor_sensible_heat'].plot()
#results['ffloor_net_radiation'].plot()

### Plot some model results

In [ ]:
sim = 0  # in this demo, we have only one simulation (i.e. only one parameter set was used)

# grid variables for plotting
t = results.date  # time
zc = results.canopy_z  # height above ground [m]
zs = results.soil_z  # depth of soil; shown negative [m]

### Soil temperature and moisture

- computed using pyAPES.soil - package submodels 'Water' and 'Heat'
- model state at end of simulation is in: Model.soil.heat & Model.soil.water

In [ ]:
# model state (last timestep)

#soilheat = Model.soil.heat
#vars(soilheat)  # list instance variables

#soilwater = Model.soil.water
#vars(soilwater) 

In [ ]:
import numpy as np
var = ['soil_temperature', 'soil_volumetric_water_content']

lyrs = [0, 1, 2, 3, 4] # five top layers
#depths = np.array2string(np.asarray(zs[lyrs]), precision=1, separator=', ')
depths = ['{:.2f} m'.format(k) for k in zs[lyrs]]

fig, ax = plt.subplots(2, 1, figsize=(10,7))

k = 0
for v in var:
    ax[k].plot(t, results[v][:,sim,lyrs], label=depths)
    ax[k].set_ylabel(results[v].attrs['units'])
    ax[k].tick_params(axis='x', labelrotation = 20)
    ax[k].legend(fontsize=8)
    k += 1

# vertical profile at last timestep
fig, ax = plt.subplots(1, 2) #figsize=(10,5))

k = 0
for v in var:
    ax[k].plot(results[v][-1,sim,:], zs)
    ax[k].set_xlabel(results[v].attrs['units'])
    ax[0].set_ylabel('depth (m)')
    k += 1

### Ecosystem-scale fluxes

- ecosystem - atm. fluxes represent the integrated sinks / sources in soil (soil-module), forestfloor (bottomlayer-module) and vegetation (planttype&canopy -modules)
- comparable to ecosystem - atmosphere exchange
- affected by current model forcing and sub-model instance state (e.g. Planttype and Canopy LAI, phenology, soil temperature, moisture etc.)


In [ ]:

var = ['forcing_air_temperature', 'forcing_par', 'canopy_NEE', 'canopy_GPP', 
       'canopy_Reco', 'canopy_Rnet', 'canopy_SH', 'canopy_LE', 'ffloor_ground_heat']

fig, ax = plt.subplots(3, 1, figsize=(8,12), sharex=True)

# temperature & par
ax[0].plot(t, results[var[0]][:,sim], 'k--', label=var[0])
ax[0].set_ylabel(results[var[0]].attrs['units'])
ax[0].tick_params(axis='x', labelrotation = 20)
axb = ax[0].twinx()

axb.plot(t, results[var[1]][:,sim], 'r-', label=var[1]) # Par
axb.set_ylabel(results[var[0]].attrs['units'])

# CO2 fluxes
for v in var[2:5]:
    ax[1].plot(t, results[v][:,sim], label=v)

ax[1].set_ylabel('umol m-2 s-1')
ax[1].tick_params(axis='x', labelrotation = 20)
ax[1].legend(fontsize=8)

# energy fluxes
for v in var[5:]:
    ax[2].plot(t, results[v][:,sim], label=v)

ax[2].set_ylabel('W m-2')
ax[2].tick_params(axis='x', labelrotation = 20)
ax[2].legend(fontsize=8)



### Ecosystem radiation balance

- net radiation (Rnet) is the sum of net shortwave (SWnet = incoming - reflected) and net longwave (LWnet = incoming - emitted) radiation at canopy top
- computed via models in pyAPES.microclimate.radiation, called iteratively from pyAPES.canopy.mlm_canopy to account for canopy structure and leaf & forest floor temperature
- ecosystem albedo can be computed from radiation profiles at uppermost grid point

In [ ]:
# net radiation components at canopy top
var = ['canopy_Rnet','canopy_SWnet', 'canopy_LWnet']
profs = ['canopy_par_down', 'canopy_par_up', 'canopy_nir_down','canopy_nir_up','canopy_lw_down','canopy_lw_up']

fig, ax = plt.subplots(3, 1, figsize=(8,12), sharex=True)

for v in var:
    ax[0].plot(t, results[v][:, sim], label=v)
ax[0].set_ylabel('W m-2')
ax[0].tick_params(axis='x', labelrotation = 20)
ax[0].legend(fontsize=8)    

# lets plot also partitioning of canopy_SWnet and canopy_LWnet.
# -1 is the index of uppermost gridpoint

for v in ['canopy_par_down', 'canopy_nir_down','canopy_lw_down']: # downward
    ax[1].plot(t, results[v][:,sim,-1], '-', label=v)
for v in ['canopy_par_up', 'canopy_nir_up','canopy_lw_up']: # upward
    ax[1].plot(t, results[v][:,sim,-1], '--', label=v)
ax[1].set_ylabel('W m-2')
ax[1].tick_params(axis='x', labelrotation = 20)
ax[1].legend(fontsize=8)

# canopy albedo
eps = 1e-16

# fraction of par on total SW
f_par = results['canopy_par_down'][:,sim,-1] / (results['canopy_par_down'][:,sim,-1] + results['canopy_nir_down'][:,sim,-1] + eps)

alb_par = results['canopy_par_up'][:,sim,-1] / (results['canopy_par_down'][:,sim,-1] + eps)
alb_nir = results['canopy_nir_up'][:,sim,-1] / (results['canopy_nir_down'][:,sim,-1] + eps)
alb_sw = f_par * alb_par + (1 - f_par) * alb_nir
alb_sw = np.maximum(0, np.minimum(1.0, alb_sw))

ax[2].plot(t, alb_sw, '-', label='SW albedo')
ax[2].plot(t, alb_par, '-', label='Par albedo')
ax[2].plot(t, alb_nir, '-', label='Nir albedo')

ax[2].tick_params(axis='x', labelrotation = 20)
ax[2].legend(fontsize=8)

### Microclimatic gradients within the canopy

- sub-models pyAPES.microclimate.Micromet & pyAPES.radiation.Radiation
- mean wind speed & friction velocity (momentum flux) profiles depend on momentum absorption in the canopy, which is proportional to leaf-area density (*lad*) profile. Canopy *lad* is sum of *lad* of each PlantType instance
- SW absorption depends on *lad* and leaf optical properties
- air-space scalar profiles (*T*, *H2O* & *CO2*) in steady-state with respective sink/source profile using 1st-order closure (K-theory)

Let's compute average profiles for daytime hours 10:00 -- 16:00 under conditions when canopy is dry


In [ ]:
#par = results['forcing_par'][:,sim] # use PAR as criteria for night / day
ix = np.where((results.date.dt.hour>=10) & (results.date.dt.hour<=16) & (results['canopy_interception_storage'] <= eps))
print(ix)

rad = ['canopy_par']
mmet = ['canopy_wind_speed', 'canopy_friction_velocity', 'canopy_temperature', 'canopy_co2', 'canopy_h2o']

plt.figure('scalarprofiles')
n = 1
for v in var:
    plt.subplot(2,2,n)
    x = results[v][:, sim, :] - results[v][:, sim, -1] # s - s_ref
    xm = np.mean(x, axis=0)
    xmn = np.mean(x.loc[par < 20, :], axis=0) # night
    xmd = np.mean(x.loc[par > 200, :], axis=0) # day
    plt.plot(x, zcm, 'k', alpha=0.1)
    plt.plot(xmd, zc, 'r', label='day'); plt.plot(xmn, zc, 'b', label='night')
    plt.xlabel(results[v].attrs['units']); plt.ylabel(zc.attrs['units'])
    n += 1

plt.legend()


In [ ]:

ix = np.where((results.date.dt.hour>=10) & (results.date.dt.hour<=17))
print(ix)

ix = np.where((results.date.dt.hour>=10) & (results.date.dt.hour<=16) & (results['canopy_interception_storage'] <= eps))
print(ix)

In [ ]:
results.close()